In [ ]:
import os
import json
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics import (
    roc_curve, auc,
    accuracy_score, f1_score,
    precision_score, recall_score,
    confusion_matrix
)
import matplotlib.pyplot as plt
from sklearn.preprocessing import normalize

In [ ]:
EMBED_MODEL      = "Qwen/Qwen3-Embedding-8B"   
EMBED_BATCH_SIZE = 16                            
EMBED_DIM        = None                         

TASK_INSTRUCTION = (
    "Given a counseling client's profile and presenting situation, "
    "retrieve the profile and situation describing the same client."
)
QUERY_PROMPT = f"Instruct: {TASK_INSTRUCTION}\nQuery:"

model = SentenceTransformer(
    EMBED_MODEL,
    truncate_dim=EMBED_DIM,
)

def embed_texts(texts, is_query):
    emb = model.encode(
        texts,
        prompt=QUERY_PROMPT if is_query else None,
        batch_size=EMBED_BATCH_SIZE,
        convert_to_numpy=True,
        show_progress_bar=True,
        normalize_embeddings=False,
    )
    return np.asarray(emb, dtype=np.float32)

PROFILES_DIR  = "path_to_extracted_demographics"                   
SITUATION_KEY = "situation of the client"    

_DROP_VALUES = {"cannot be identified", "not enough information",
                "unknown", "n/a", "none", ""}

def standardize_profile(profile):
    if not profile:
        return {}
    out = {}
    for k, v in profile.items():
        if v is None:
            continue
        if isinstance(v, str) and v.strip().lower() in _DROP_VALUES:
            continue
        out[k] = v
    return out

def load_session_profiles(session_num, profiles_dir=PROFILES_DIR):
    path = os.path.join(profiles_dir, f"session_{session_num}.json")
    if not os.path.exists(path):
        return None, None
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    real      = standardize_profile(data.get("actual profile", {}))
    extracted = standardize_profile(data.get("synthetic profile", {}))
    return real, extracted

def format_profile_situation(profile, situation):
    if profile:
        attrs = "; ".join(f"{k}: {v}" for k, v in profile.items())
    else:
        attrs = "(no identifiable attributes)"
    situation = (situation or "").strip()
    return f"Profile: {attrs}\nSituation: {situation}"

PROFILE_ATTRS = ["name", "gender", "age", "occupation", "marital status"]

def real_profile_from_parquet(parquet_idx):
    prof = json.loads(profiles[parquet_idx])
    return standardize_profile({k: prof[k] for k in PROFILE_ATTRS if k in prof})

In [ ]:
with open("../valid_indices.json", "r") as f:
    valid_indices = json.load(f)

In [ ]:
with open('path_to_extracted_situation', 'r') as file:
    situation_1 = json.load(file)

In [ ]:
df = pd.read_parquet("../eeyore-data.parquet")

In [6]:
profiles = df['profile'].tolist()

In [ ]:
extracted_text_list = []   # extracted profile + extracted situation
real_text_list      = []   # real (actual) profile + real situation
session_list        = []

In [8]:
for i in range(550):
    if ("session_" + str(i)) in situation_1.keys() and (i - 1) in valid_indices:
        # extracted ("synthetic") profile from session_<i>.json;
        # real profile from the parquet demographic fields.
        _, extracted_profile = load_session_profiles(i)
        if extracted_profile is None:        # session_<i>.json missing -> skip comparison
            continue
        real_profile = real_profile_from_parquet(i - 1)

        profile_i           = json.loads(profiles[i - 1])
        extracted_situation = situation_1["session_" + str(i)]
        real_situation      = profile_i[SITUATION_KEY]

        extracted_text_list.append(
            format_profile_situation(extracted_profile, extracted_situation))
        real_text_list.append(
            format_profile_situation(real_profile, real_situation))
        session_list.append("session_" + str(i))

In [ ]:
embeddings_extracted = embed_texts(extracted_text_list, is_query=True)

In [ ]:
embeddings_real = embed_texts(real_text_list, is_query=False)

In [11]:
embeddings1 = normalize(embeddings_extracted)
embeddings2 = normalize(embeddings_real)

In [12]:
similarity_matrix = np.matmul(embeddings1, embeddings2.T)

In [13]:
n = similarity_matrix.shape[0]

In [14]:
# Labels and scores
scores = []
labels = []

In [15]:
for i in range(n):
    for j in range(n):
        scores.append(similarity_matrix[i, j])
        if i == j:
            labels.append(1)  # positive pair
        else:
            labels.append(0)  # negative pair

In [16]:
scores = np.array(scores)

In [17]:
labels = np.array(labels)

In [18]:
fpr, tpr, thresholds = roc_curve(labels, scores)
roc_auc = auc(fpr, tpr)

In [ ]:
# ---- TPR at specific low FPR thresholds ----
target_fprs = [0.1, 0.05, 0.01]

print(f"{'FPR Target':<15} {'Actual FPR':<15} {'TPR':<10} {'Threshold':<12}")
print("-" * 55)
for target in target_fprs:
    # Find the index where fpr <= target (closest without exceeding)
    valid_idx = np.where(fpr <= target)[0]
    if len(valid_idx) == 0:
        print(f"{target:<15.2f} {'N/A':<15} {'N/A':<10} {'N/A':<12}")
        continue
    idx = valid_idx[-1]  # largest fpr that is still <= target
    print(f"{target:<15.2f} {fpr[idx]:<15.4f} {tpr[idx]:<10.4f} {thresholds[idx]:<12.4f}")

In [ ]:
# ---- Zoomed ROC Curve at low FPR region (log scale) ----
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Left: Full ROC for reference ---
ax1 = axes[0]
ax1.plot(fpr, tpr, color='steelblue', lw=2, label=f'ROC curve (AUC = {roc_auc:.4f})')
ax1.plot([0, 1], [0, 1], 'k--', lw=1, label='Random classifier')
for target in [0.1, 0.05, 0.01]:
    valid_idx = np.where(fpr <= target)[0]
    if len(valid_idx) == 0:
        continue
    idx = valid_idx[-1]
    ax1.plot(fpr[idx], tpr[idx], 'o', markersize=7,
             label=f'FPR≤{target}: TPR={tpr[idx]:.3f}')
ax1.set_xlabel('False Positive Rate')
ax1.set_ylabel('True Positive Rate')
ax1.set_title('Full ROC Curve')
ax1.legend(loc='lower right', fontsize=8)
ax1.grid(True, alpha=0.3)

# --- Right: Zoomed ROC with log-scale FPR ---
ax2 = axes[1]
# Filter to low FPR region (fpr > 0 to allow log scale)
mask = (fpr > 0) & (tpr > 0)
fpr_log = fpr[mask]
tpr_log = tpr[mask]

ax2.plot(fpr_log, tpr_log, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.4f})')
for target in [0.1, 0.05, 0.01]:
    valid_idx = np.where(fpr <= target)[0]
    if len(valid_idx) == 0:
        continue
    idx = valid_idx[-1]
    if fpr[idx] > 0:
        ax2.axvline(x=fpr[idx], color='gray', linestyle=':', alpha=0.6, lw=1)
        ax2.plot(fpr[idx], tpr[idx], 'o', markersize=8,
                 label=f'FPR≤{target}: TPR={tpr[idx]:.3f}')

ax2.set_xscale('log')
ax2.set_yscale('log')
ax2.set_xlim([0.005, 0.15])  # zoom into low FPR region
ax2.set_xlabel('False Positive Rate (log scale)')
ax2.set_ylabel('True Positive Rate (log scale)')
ax2.set_title('Zoomed ROC Curve — Low FPR Region (Log-Log Scale)')
ax2.legend(loc='lower right', fontsize=8)
ax2.grid(True, which='both', alpha=0.3)

plt.tight_layout()
plt.savefig("path_plot_file", dpi=300, bbox_inches='tight')
plt.show()

In [21]:
j_scores = tpr - fpr
best_index = np.argmax(j_scores)
best_threshold = thresholds[best_index]

In [ ]:
print(f"Best threshold (Youden's J): {best_threshold:.4f}")
print(f"TPR at best threshold: {tpr[best_index]:.4f}")
print(f"FPR at best threshold: {fpr[best_index]:.4f}")

In [23]:
# ---- choose a threshold ----
threshold = best_threshold
preds = (scores >= threshold).astype(int)

In [ ]:
##### ---- basic metrics ----
accuracy = accuracy_score(labels, preds)
precision = precision_score(labels, preds)
recall = recall_score(labels, preds)   # same as TPR
f1 = f1_score(labels, preds)

# ---- confusion matrix ----
tn, fp, fn, tp = confusion_matrix(labels, preds).ravel()

# ---- derived metrics ----
fnr = fn / (fn + tp)   # False Negative Rate
tnr = tn / (tn + fp)   # True Negative Rate (Specificity)

print({
    "AUC": roc_auc,
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall (TPR)": recall,
    "F1": f1,
    "FNR": fnr,
    "TNR": tnr
})